### EasyOCR

In [2]:
import easyocr

#Carga del modelo de lengua
reader = easyocr.Reader(['es']) 

#Reconocimiento de una imagen
res = reader.readtext('matricula.jpg')

for (bbox, text, prob) in res:
    # Coordenadas en orden 
    (top_left, top_right, bottom_right, bottom_left) = bbox
    print(f'\nTexto: {text}\nProbabilidad: {prob:.2f}\nContenedor: {tuple(map(int, top_left)),tuple(map(int, bottom_right))}')


#Con restricción de caracteres reconocibles
#result = reader.readtext('toy.tif', allowlist ='0123456789')


Texto: 072L HPH
Probabilidad: 0.42
Contenedor: ((11, 1), (126, 38))


### Tesseract

In [ ]:
# Tesseract
import cv2
import pytesseract
from pytesseract import Output

# Previamente debes descargar los ejecutables
# Si la ruta de Tesseract no está en el PATH, ruta al ejecutable
pytesseract.pytesseract.tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract'

# Lenguajes disponibles
print(pytesseract.get_languages(config=''))

#Cargo imagen y ocnvierto a RGB
img = cv2.imread('ocr_test.tif') 

if img is not None:
    #Convierte a RGB antes de procesar
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    #Texto localizado
    print(pytesseract.image_to_string(img))

    #Texto y localización en imagen de cada palabra
    d = pytesseract.image_to_data(img_rgb, output_type=Output.DICT)

    n_boxes = len(d['text'])
    for i in range(n_boxes):
        #Nivel de confianza
        if int(d['conf'][i]) > 60:
            text = d['text'][i]
            conf = d['conf'][i]
            (x, y, w, h) = (d['left'][i], d['top'][i], d['width'][i], d['height'][i])
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

            print(f'Texto: {text} ({conf:.2f}%)\nContenedor: {x,y,x+w,y+h}')

    cv2.imshow('img', img_rgb)
    cv2.waitKey(-1)

   

else:
    print('Error de imagen')

### PaddleOCR

In [ ]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang='en') 
img_path = 'matricula.jpg'
res = ocr.ocr(img_path, cls=True)

texts  = [x[1][0] for x in res[0]]
scores = [x[1][1] for x in res[0]]
print(texts, scores)


download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to C:\Users\luisp/.paddleocr/whl\det\en\en_PP-OCRv3_det_infer\en_PP-OCRv3_det_infer.tar


100%|██████████| 4.00M/4.00M [00:20<00:00, 191kiB/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to C:\Users\luisp/.paddleocr/whl\rec\en\en_PP-OCRv4_rec_infer\en_PP-OCRv4_rec_infer.tar


100%|██████████| 10.2M/10.2M [00:04<00:00, 2.32MiB/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to C:\Users\luisp/.paddleocr/whl\cls\ch_ppocr_mobile_v2.0_cls_infer\ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2.19M/2.19M [00:16<00:00, 132kiB/s] 

[2025/11/03 17:35:01] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\luisp/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\luisp/.paddleocr/whl\\rec\\en\\en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_len

[2025/11/03 17:35:04] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.15498137474060059
[2025/11/03 17:35:04] ppocr DEBUG: cls num  : 1, elapsed : 0.05655980110168457
[2025/11/03 17:35:04] ppocr DEBUG: rec_res num  : 1, elapsed : 0.14200329780578613
['0724HPH'] [0.9771336317062378]


### SmolVLM

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
Device = "cpu"  # or "cpu"

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Instruct")
model = AutoModelForImageTextToText.from_pretrained("HuggingFaceTB/SmolVLM-Instruct",
                                                dtype=torch.bfloat16,
                                                _attn_implementation="flash_attention_2" if Device == "cuda" else "eager").to(Device)


In [ ]:
from PIL import Image
from transformers.image_utils import load_image
from matplotlib import pyplot as plt

# Load images

image_1 = Image.open('matricula.jpg')
# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Can you give me the text in the license plate of the image?"}
        ]
    },
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image_1], return_tensors="pt")
inputs = inputs.to(Device)

generated_ids = model.generate(**inputs, max_new_tokens=10)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

plt.imshow(image_1)
plt.axis('off')
plt.show()
print(generated_texts[0])

In [ ]:
import os, csv, cv2
from collections import defaultdict
from ultralytics import YOLO
import pytesseract
import easyocr
from paddleocr import PaddleOCR

# ---------- RUTAS ----------
VIDEO_IN  = r"C:\Users\luisp\Desktop\VC\prac1\P4\videos\C0142.mp4"
VIDEO_OUT = r"C:\Users\luisp\Desktop\VC\prac1\P4\outputs\video_annotado2.mp4"
CSV_OUT   = r"C:\Users\luisp\Desktop\VC\prac1\P4\outputs\detecciones2.csv"

# ---------- MODELOS ----------
detector = YOLO("yolo11n.pt")  # personas/vehículos
plate_model = YOLO(r"C:\Users\luisp\Desktop\VC\prac1\P4\runs\detect\plates_s_1280_rect\weights\best.pt")  # matrículas

# ---------- OPCIONES ----------
TARGET_CLASSES = {"person", "car", "motorbike", "bus", "truck"}
TRACKER = "bytetrack.yaml"     # o BoT-SORT por defecto
DET_CONF = 0.25                # detector general
PLATE_CONF = 0.35              # detector matrículas (ajusta 0.30–0.50)
PLATE_IOU = 0.50
PLATE_IMGSZ = 1280            # 640–1280 según VRAM/recall
PLATE_ONLY_BOTTOM_BAND = True
BOTTOM_FRAC = 0.40             # 40% inferior del vehículo
EXTRA_BAND_UP = 0.05           # +5% hacia arriba por seguridad
MIN_PLATE_AREA = 700           # píxeles
PLATE_AR_MIN, PLATE_AR_MAX = 2.0, 5.0

os.makedirs(os.path.dirname(VIDEO_OUT), exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUT), exist_ok=True)

# ---------- VIDEO IO ----------
cap = cv2.VideoCapture(VIDEO_IN)
if not cap.isOpened():
    raise FileNotFoundError(f"No puedo abrir el vídeo: {VIDEO_IN}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (w, h))

# ---------- CSV ----------
csv_f = open(CSV_OUT, "w", newline="", encoding="utf-8")
cw = csv.writer(csv_f)
cw.writerow([ 
    "frame", "tipo_objeto", "confianza", "id_tracking", "x1", "y1", "x2", "y2",
    "matricula_flag", "conf_matricula", "mx1", "my1", "mx2", "my2", "texto_matricula"
])

# ---------- OCRs ----------
# Iniciar OCRs
ocr_tesseract = pytesseract
ocr_easy = easyocr.Reader(['en'])
ocr_paddle = PaddleOCR(use_angle_cls=True, lang='en')

# ---------- LOOP ----------
seen_ids = defaultdict(set)
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    # Tracking del detector general (1 resultado por frame)
    gen = detector.track(
        source=frame, stream=True, persist=True,
        tracker=TRACKER, conf=DET_CONF, verbose=False
    )
    try:
        res = next(gen)
    except StopIteration:
        res = None

    if res is None or res.boxes is None or len(res.boxes) == 0:
        writer.write(frame)
        frame_idx += 1
        continue

    names = detector.model.names
    boxes = res.boxes

    for b in boxes:
        cls_id = int(b.cls[0].item())
        conf   = float(b.conf[0].item())
        name   = names.get(cls_id, str(cls_id))

        if name not in TARGET_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
        track_id = int(b.id[0].item()) if b.id is not None else -1
        seen_ids[name].add(track_id)

        color = (0, 255, 0) if name != "person" else (0, 200, 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{name} {conf:.2f} ID:{track_id}",
                    (x1, max(0, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # -------- MATRÍCULAS SOLO EN VEHÍCULOS (con franja inferior) --------
        plate_flag, plate_conf, plate_box, plate_text = 0, 0.0, (0,0,0,0), ""

        if name in {"car", "motorbike", "bus", "truck"}:
            vx1, vy1, vx2, vy2 = x1, y1, x2, y2

            # ROI = franja inferior del vehículo (+5% hacia arriba)
            if PLATE_ONLY_BOTTOM_BAND:
                vh = vy2 - vy1
                band_top = vy2 - int(BOTTOM_FRAC * vh)
                band_top = band_top - int(EXTRA_BAND_UP * vh)
                rx1, ry1, rx2, ry2 = vx1, band_top, vx2, vy2
            else:
                rx1, ry1, rx2, ry2 = vx1, vy1, vx2, vy2

            # Limitar ROI a los bordes del frame
            rx1 = max(0, rx1); ry1 = max(0, ry1)
            rx2 = min(w, rx2); ry2 = min(h, ry2)

            if rx2 > rx1 and ry2 > ry1:
                crop = frame[ry1:ry2, rx1:rx2]

                # Predicción de matrículas
                p = plate_model.predict(
                    source=crop, conf=PLATE_CONF, iou=PLATE_IOU,
                    imgsz=PLATE_IMGSZ, max_det=3, verbose=False
                )

                if p and len(p[0].boxes) > 0:
                    pb = max(p[0].boxes, key=lambda bb: float(bb.conf[0].item()))
                    px1, py1, px2, py2 = map(int, pb.xyxy[0].tolist())
                    pconf = float(pb.conf[0].item())

                    # —— Reproyección CORRECTA a coords del frame (usar rx1/ry1) —— 
                    mx1 = rx1 + px1
                    my1 = ry1 + py1
                    mx2 = rx1 + px2
                    my2 = ry1 + py2

                    # Filtros geométricos
                    wpl, hpl = mx2 - mx1, my2 - my1
                    ar = wpl / max(1, hpl)
                    if (wpl * hpl) >= MIN_PLATE_AREA and PLATE_AR_MIN <= ar <= PLATE_AR_MAX:
                        plate_flag, plate_conf, plate_box = 1, pconf, (mx1, my1, mx2, my2)
                        cv2.rectangle(frame, (mx1, my1), (mx2, my2), (255, 0, 0), 2)
                        cv2.putText(frame, f"PLATE {pconf:.2f}",
                                    (mx1, max(0, my1 - 6)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

                        # OCR para extraer texto de la matrícula
                        plate_text = ""

                        # Usar Tesseract para OCR
                        plate_text = pytesseract.image_to_string(crop, config='--psm 6')

                        # Usar EasyOCR para OCR
                        ocr_result = ocr_easy.readtext(crop)
                        if ocr_result:
                            plate_text = max(ocr_result, key=lambda x: len(x[1]))[1]

                        # Usar PaddleOCR para OCR
                        ocr_result = ocr_paddle.ocr(crop, cls=True)
                        if ocr_result:
                            plate_text = ''.join([res[1][0] for res in ocr_result[0]])

        mx1, my1, mx2, my2 = plate_box
        cw.writerow([
            frame_idx, name, f"{conf:.3f}", track_id, x1, y1, x2, y2,
            plate_flag, f"{plate_conf:.3f}", mx1, my1, mx2, my2, plate_text
        ])

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
csv_f.close()

print("\nConteo de IDs únicos (aprox):")
for k, ids in seen_ids.items():
    ids.discard(-1)
    print(f"  {k}: {len(ids)}")
print(f"\nVídeo anotado: {VIDEO_OUT}")
print(f"CSV: {CSV_OUT}")
